# BERT4Rec — Full-Rank Evaluation trên MovieLens-1M

Đánh giá model `bert4rec_best.pth` theo giao thức **full-rank** (toàn bộ catalog item).

**Metrics:** `HR@K`, `NDCG@K`, `MRR@K`  
**Split:** Leave-one-out — item cuối cùng làm test ground-truth  
**History masking:** Loại bỏ các item đã xuất hiện trong train trước khi rank


## 0. Cài đặt thư viện

In [1]:
!pip install torch pandas numpy tqdm --quiet


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Cấu hình — chỉnh sửa các đường dẫn tại đây

In [2]:
# ── Đường dẫn file ───────────────────────────────────────────────────────
RATINGS_PATH  = r"F:\1_REL\Reinforcement-learning-for-Recommendation\Data_Movielens_1m\ml-1m\ratings.dat"   # ratings.dat của MovieLens-1M
MODEL_PATH    = r"F:\1_REL\Reinforcement-learning-for-Recommendation\saved_models\bert4rec_best.pth"             # file checkpoint
MODEL_SRC     = r"F:\1_REL\Reinforcement-learning-for-Recommendation\Bert4Rec_model.py"            # file kiến trúc model

# ── Siêu tham số đánh giá ────────────────────────────────────────────────
TOPK_LIST     = [1, 5, 10, 20]   # các giá trị K cần tính
BATCH_SIZE    = 256               # giảm xuống 64 nếu OOM
MAX_SEQ_LEN   = 200               # phải khớp với MAX_LEN khi train
MIN_SEQ_LEN   = 3                 # bỏ user có chuỗi ngắn hơn giá trị này
DEVICE        = "auto"            # 'auto' | 'cuda' | 'cpu'


## 2. Import thư viện

In [3]:
import importlib.util
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")


PyTorch  : 2.5.1+cu121
CUDA     : True


## 3. Tải kiến trúc BERT4Rec

In [4]:
def load_bert4rec_class(model_src_path: str):
    """Import BERT4Rec class trực tiếp từ đường dẫn file .py."""
    spec   = importlib.util.spec_from_file_location("bert4rec_module", model_src_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.BERT4Rec

BERT4Rec = load_bert4rec_class(MODEL_SRC)
print(f"✅ BERT4Rec class loaded từ: {MODEL_SRC}")


✅ BERT4Rec class loaded từ: F:\1_REL\Reinforcement-learning-for-Recommendation\Bert4Rec_model.py


## 4. Load & tiền xử lý MovieLens-1M

In [5]:
def load_ml1m(ratings_path: str):
    """
    Đọc ratings.dat, sắp xếp theo (user, timestamp), trả về:
        user_sequences : dict  user_id → list[item_idx]  (index bắt đầu từ 1)
        n_items        : int   số item duy nhất
        item2idx       : dict  item gốc → item index
    """
    print(f"[Data] Đọc {ratings_path} ...")
    df = pd.read_csv(
        ratings_path,
        sep="::",
        engine="python",
        header=None,
        names=["user", "item", "rating", "timestamp"],
    )

    # Sắp xếp theo thời gian
    df.sort_values(["user", "timestamp"], inplace=True)

    # Re-index item từ 1 (0 = PAD)
    unique_items   = sorted(df["item"].unique())
    item2idx       = {item: idx + 1 for idx, item in enumerate(unique_items)}
    n_items        = len(unique_items)
    df["item_idx"] = df["item"].map(item2idx)

    user_sequences = (
        df.groupby("user")["item_idx"]
        .apply(list)
        .to_dict()
    )

    print(f"[Data] Users: {len(user_sequences):,}  |  Items: {n_items:,}  |  Interactions: {len(df):,}")
    return user_sequences, n_items, item2idx


user_seqs, n_items, item2idx = load_ml1m(RATINGS_PATH)


[Data] Đọc F:\1_REL\Reinforcement-learning-for-Recommendation\Data_Movielens_1m\ml-1m\ratings.dat ...
[Data] Users: 6,040  |  Items: 3,706  |  Interactions: 1,000,209


## 5. Leave-one-out split

In [6]:
def split_sequences(user_sequences: dict, min_len: int = 3):
    """
    Leave-one-out:
        train_seqs[u] = seq[:-1]   (lịch sử để feed model)
        test_items[u] = seq[-1]    (item ground-truth)
    Bỏ user có chuỗi < min_len.
    """
    train_seqs = {}
    test_items = {}
    skipped    = 0
    for u, seq in user_sequences.items():
        if len(seq) < min_len:
            skipped += 1
            continue
        train_seqs[u] = seq[:-1]   # bỏ item cuối ra test
        test_items[u] = seq[-1]    # ground-truth
    print(f"[Split] Eval users: {len(train_seqs):,}  (bỏ {skipped} user ngắn)")
    return train_seqs, test_items


train_seqs, test_items = split_sequences(user_seqs, min_len=MIN_SEQ_LEN)


[Split] Eval users: 6,040  (bỏ 0 user ngắn)


## 6. Load checkpoint

In [7]:
# ── Device ──────────────────────────────────────────────────────────────────
if DEVICE == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(DEVICE)
print(f"[Device] {device}")

# ── Load checkpoint ───────────────────────────────────────────────────────────
print(f"[Model] Load checkpoint: {MODEL_PATH}")
ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)

# Hỗ trợ checkpoint dạng dict{'state_dict': ...} hoặc state_dict trực tiếp
if isinstance(ckpt, dict) and "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
elif isinstance(ckpt, dict) and "output_bias" in ckpt:
    state_dict = ckpt
else:
    state_dict = None   # toàn bộ ckpt là model object

# Auto-detect n_items từ output_bias
if state_dict is not None and "output_bias" in state_dict:
    n_items_ckpt = state_dict["output_bias"].shape[0]
    if n_items_ckpt != n_items:
        print(f"[WARNING] n_items checkpoint ({n_items_ckpt}) ≠ data ({n_items}). Dùng checkpoint value.")
        n_items = n_items_ckpt

# Khởi tạo model
model = BERT4Rec(
    n_items             = n_items,
    max_seq_length      = MAX_SEQ_LEN,
    hidden_size         = 64,
    n_layers            = 2,
    n_heads             = 2,
    inner_size          = 256,
    hidden_dropout_prob = 0.2,
    attn_dropout_prob   = 0.2,
    hidden_act          = "gelu",
    mask_ratio          = 0.2,
    loss_type           = "CE",
)

if state_dict is not None:
    model.load_state_dict(state_dict, strict=True)
else:
    model = ckpt

model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"[Model] n_items={n_items:,} | params={n_params:,} | max_seq_len={MAX_SEQ_LEN}")
print("✅ Model loaded!")


[Device] cuda
[Model] Load checkpoint: F:\1_REL\Reinforcement-learning-for-Recommendation\saved_models\bert4rec_best.pth
[WARNING] n_items checkpoint (3707) ≠ data (3706). Dùng checkpoint value.
[Model] n_items=3,707 | params=358,203 | max_seq_len=200
✅ Model loaded!


## 7. Định nghĩa Metrics

In [8]:
def hit_at_k(rank: int, k: int) -> float:
    """HR@K: 1 nếu ground-truth item nằm trong top-K, ngược lại 0."""
    return 1.0 if rank <= k else 0.0


def ndcg_at_k(rank: int, k: int) -> float:
    """NDCG@K: 1/log2(rank+1) nếu rank <= K."""
    return 1.0 / np.log2(rank + 1) if rank <= k else 0.0


def mrr_at_k(rank: int, k: int) -> float:
    """MRR@K: 1/rank nếu rank <= K."""
    return 1.0 / rank if rank <= k else 0.0


print("✅ Metric functions defined.")


✅ Metric functions defined.


## 8. Evaluation Loop (Full-Rank)

In [9]:
def evaluate(
    model,
    train_seqs: dict,
    test_items: dict,
    n_items: int,
    max_seq_len: int,
    topk_list: list,
    device: torch.device,
    batch_size: int = 256,
) -> dict:
    """
    Full-rank evaluation. Với mỗi user:
      1. Feed train_seq vào model
      2. full_sort_predict → score toàn bộ n_items
      3. History masking — đặt score item đã xem = -inf
      4. Tính rank của ground-truth → HR, NDCG, MRR
    """
    model.eval()
    users   = list(train_seqs.keys())
    metrics = {f"{m}@{k}": 0.0 for m in ["HR", "NDCG", "MRR"] for k in topk_list}
    n_eval  = 0

    for start in tqdm(range(0, len(users), batch_size), desc="Evaluating"):
        batch_users = users[start: start + batch_size]

        # Chuẩn bị tensor
        item_seq_t   = torch.zeros(len(batch_users), max_seq_len, dtype=torch.long)
        item_seq_len = torch.zeros(len(batch_users), dtype=torch.long)

        for i, u in enumerate(batch_users):
            seq     = train_seqs[u][-max_seq_len:]   # cắt nếu quá dài
            seq_len = len(seq)
            item_seq_t[i, :seq_len] = torch.tensor(seq, dtype=torch.long)
            item_seq_len[i]         = seq_len

        item_seq_t   = item_seq_t.to(device)
        item_seq_len = item_seq_len.to(device)

        with torch.no_grad():
            scores = model.full_sort_predict(item_seq_t, item_seq_len)  # [B, n_items]
            scores = scores.cpu().numpy()

        for i, u in enumerate(batch_users):
            score_arr = scores[i].copy()          # [n_items]
            gt_item   = test_items[u]             # 1-based index

            # History masking
            history = train_seqs[u][:-1] if len(train_seqs[u]) > 1 else []
            for hist_item in history:
                if 1 <= hist_item <= n_items:
                    score_arr[hist_item - 1] = -np.inf

            gt_score = score_arr[gt_item - 1]
            rank     = int((score_arr > gt_score).sum()) + 1

            for k in topk_list:
                metrics[f"HR@{k}"]   += hit_at_k(rank, k)
                metrics[f"NDCG@{k}"] += ndcg_at_k(rank, k)
                metrics[f"MRR@{k}"]  += mrr_at_k(rank, k)
            n_eval += 1

    for key in metrics:
        metrics[key] /= n_eval
    metrics["n_eval"] = n_eval
    return metrics


print("✅ evaluate() defined.")


✅ evaluate() defined.


## 9. Chạy đánh giá

In [10]:
topk_list = sorted(set(TOPK_LIST))
print(f"[Eval] Full-rank evaluation | K = {topk_list}")

t0 = time.time()
results = evaluate(
    model       = model,
    train_seqs  = train_seqs,
    test_items  = test_items,
    n_items     = n_items,
    max_seq_len = MAX_SEQ_LEN,
    topk_list   = topk_list,
    device      = device,
    batch_size  = BATCH_SIZE,
)
elapsed = time.time() - t0
n_eval  = results.pop("n_eval")
print(f"\n⏱  Hoàn thành trong {elapsed:.1f}s  |  {n_eval:,} users")


[Eval] Full-rank evaluation | K = [1, 5, 10, 20]


Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]


⏱  Hoàn thành trong 1.2s  |  6,040 users


## 10. Kết quả

In [11]:
header = f"{'Metric':<12}" + "".join(f"  K={k:<5}" for k in topk_list)
sep    = "─" * len(header)

print(sep)
print("  BERT4Rec  —  Full-Rank Evaluation  —  MovieLens-1M")
print(sep)
print(header)
print(sep)
for metric_name in ["HR", "NDCG", "MRR"]:
    row = f"{metric_name:<12}"
    for k in topk_list:
        row += f"  {results[f'{metric_name}@{k}']:.4f} "
    print(row)
print(sep)


────────────────────────────────────────────────
  BERT4Rec  —  Full-Rank Evaluation  —  MovieLens-1M
────────────────────────────────────────────────
Metric        K=1      K=5      K=10     K=20   
────────────────────────────────────────────────
HR            0.0033   0.0161   0.0263   0.0488 
NDCG          0.0033   0.0094   0.0127   0.0183 
MRR           0.0033   0.0073   0.0086   0.0101 
────────────────────────────────────────────────


## 11. Xuất kết quả ra DataFrame & lưu file

In [12]:
rows = []
for metric_name in ["HR", "NDCG", "MRR"]:
    row = {"Metric": metric_name}
    for k in topk_list:
        row[f"@{k}"] = round(results[f"{metric_name}@{k}"], 4)
    rows.append(row)

df_results = pd.DataFrame(rows).set_index("Metric")
display(df_results)

# Lưu CSV
out_csv = "fullrank_results.csv"
df_results.to_csv(out_csv)
print(f"\n✅ Kết quả lưu tại: {out_csv}")


,@1,@5,@10,@20
Metric,,,,
HR,0.0033,0.0161,0.0263,0.0488
NDCG,0.0033,0.0094,0.0127,0.0183
MRR,0.0033,0.0073,0.0086,0.0101



✅ Kết quả lưu tại: fullrank_results.csv
